# Inferring GRN on P22 data

### This tutorial is used to demonstrate how to obtain gene regulatory network

### Before running this tutorial, you need to complement [this tutorial](https://github.com/mcllllllll/SpaDC/blob/master/Tutorials/Tutorial_P22.ipynb) first.

### There are three modules: identifying functional CREs for each gene, detecting TFs for functional CREs, and linking TF to target genes

# Module 1: identifying functional CREs for each gene

### Getting TSS annotation regions

In [1]:
import scanpy as sc
import pandas as pd
import pybedtools
import SpaDC
import os
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "./SpaDC"

tss = pd.read_csv(os.path.join(DATA_DIR, "GRN/mm10_TSS.txt"), sep='\t') ### mm10_TSS is downloaded from https://github.com/azofeifa/Tfit/blob/master/annotation_files/mm10_TSS.bed
print(tss)

### Peaks within ±250kb to TSS are treated as candidate functional peaks
tss['end'] = tss['start'] + 250000 
tss['start'] = tss['start'] - 250000

print(tss[tss['start'] < 0])
tss.loc[tss['start'] < 0, 'start'] = 0
print(tss[tss['start'] == 0])

tss.columns = [0, 1, 2, 'gene']
print(tss)

tss.to_csv(os.path.join(DATA_DIR, "GRN/mm10_TSS.bed"), sep='\t', index=False)

        chr     start       end     Gene
1      chr1   3466587   3513553   Gm1992
2      chr1   3205901   3671498     Xkr4
3      chr1   3905739   3986215  Gm37381
4      chr1   4292981   4409187    Rp1_2
5      chr1   3999557   4409241      Rp1
...     ...       ...       ...      ...
27917  chrY  90603501  90605864  Gm28300
27918  chrY  90665346  90667625  Gm28301
27919  chrY  90754513  90754821  Gm21860
27920  chrY  90784738  90816464    Erdr1
27921  chrY  90838869  90839177  Gm21748

[27916 rows x 4 columns]
        chr   start     end     Gene
26090  chrM -247249  252751   mt-Nd1
26091  chrM -246086  253914   mt-Nd2
26092  chrM -244672  255328   mt-Co1
26093  chrM -242987  257013   mt-Co2
26094  chrM -242234  257766  mt-Atp8
26095  chrM -242073  257927  mt-Atp6
26096  chrM -241393  258607   mt-Co3
26097  chrM -240541  259459   mt-Nd3
26098  chrM -240123  259877  mt-Nd4l
26099  chrM -239833  260167   mt-Nd4
26100  chrM -238258  261742   mt-Nd5
26101  chrM -236448  263552   mt-Nd6
2

### Identifying marker genes from each spatial domain detected by SpaDC

##### We focused on the marker genes

In [2]:
rna = sc.read_h5ad(os.path.join(DATA_DIR, "P22/ATAC/adata_RNA.h5ad"))
# Load P22 result generated from Tutorial_P22.ipynb
result = sc.read_h5ad(os.path.join(DATA_DIR, "P22/ATAC/p22_ATAC_result.h5ad"))

cluster_annotation = {
    '1-cp':'cp', 
    '2-ctx':'ctx', 
    '3-ctx':'ctx', 
    '4-ctx':'ctx', 
    '5-ctx':'ctx', 
    '6-acb':'acb', 
    '7-ccg':'ccg', 
    '8-acb':'acb', 
    '9-ctx':'ctx', 
    '10-ls':'ls', 
    '11-aca':'aca', 
    '12-ctx':'ctx', 
    '13-cp':'cp', 
    '14-ndb':'ndb', 
    '15-aca':'aca', 
    '16-vl':'vl', 
    '17-aco':'aco', 
    '18-lpo':'lpo'
}

result.obs['SpaDC'] = result.obs['SpaDC'].map(cluster_annotation).astype('category')
rna.obs['SpaDC'] = result.obs['SpaDC'].values

adata = rna.copy()
adata.raw = adata

sc.tl.rank_genes_groups(
    adata,
    groupby='SpaDC',
    method='wilcoxon',       
    n_genes=adata.shape[1],
    use_raw=True,
    key_added='rg_SpaDC'
)

df = sc.get.rank_genes_groups_df(adata, key='rg_SpaDC', group=None)
df = df[df['logfoldchanges'] > 0]

top10 = (df.sort_values(['group','pvals_adj','scores'], ascending=[True, True, False])
           .groupby('group', as_index=False)
           .head(10))

top10_dict = top10.groupby('group')['names'].apply(list).to_dict()
df_top10 = pd.DataFrame([
    {"group": g, "marker_genes": ",".join(genes)}
    for g, genes in top10_dict.items()
])
df_top10.to_csv(os.path.join(DATA_DIR, "GRN/top10_marker_genes.csv.csv"), index=False)

marker_genes = top10['names'].unique().tolist()
print(marker_genes)
print(len(marker_genes))

tss = pd.read_csv(os.path.join(DATA_DIR, "GRN/mm10_TSS.bed"), sep='\t')
print(tss)

tss = tss[tss['gene'].isin(marker_genes)]
print(tss)

tss.to_csv(os.path.join(DATA_DIR, "GRN/marker_genes_TSS.bed"),sep='\t', index=False)

['Nfix', 'C1ql3', 'Mef2c', 'Chd3', 'Camk2a', 'Sparcl1', 'Tenm2', 'Zbtb16', 'Tcf4', 'Mical2', 'Gng7', 'Pde10a', 'Cpne5', 'Zbtb20', 'Meis2', 'Meg3', 'Gria1', 'Bcl11b', 'Rgs9', 'Gnal', 'Plp1', 'Mbp', 'Mobp', 'Mal', 'Mag', 'mt.Rnr1', 'Scd2', 'Nfasc', 'Sox2ot', 'Map4k4', 'mt.Rnr2', 'Fth1', 'Kif5a', 'Qk', 'Smpd3', 'Atp2b1', 'Foxp1', 'Adcy5', 'Syndig1l', 'Ppp1r1b', 'Nrgn', 'X3110035E14Rik', 'Slc17a7', 'Olfm1', 'Pgm2l1', 'Gm26924', 'Camk2n1', 'Snap25', 'Atp1b1', 'Rreb1', 'Negr1', 'Btg1', 'Wdfy3', 'Col11a1', 'Malat1', 'Son', 'Nrxn3', 'Dgkg', 'Cacna2d2', 'Ahi1', 'Atp2b4', 'Snhg11', 'Clmn', 'Col25a1', 'Gfra1', 'Gprasp2', 'Nap1l5', 'X5330434G04Rik', 'Nefh', 'Gad1', 'Ece2', 'Scn9a', 'Tmem130', 'Ccnd2', 'Sox4', 'Ptma', 'Nfib', 'Chd7', 'Msi2', 'Slc1a2', 'Sox2']
81
          0         1         2     gene
0      chr1   3216587   3716587   Gm1992
1      chr1   2955901   3455901     Xkr4
2      chr1   3655739   4155739  Gm37381
3      chr1   4042981   4542981    Rp1_2
4      chr1   3749557   4249557    

### Extracting ATAC data with candidated functional peaks

In [3]:
atac = sc.read_h5ad(os.path.join(DATA_DIR, "P22/ATAC/adata_peaks_normalized.h5ad"))
index = atac.var_names
index = pd.DataFrame(x.split('-') for x in index)
index.to_csv(os.path.join(DATA_DIR, "GRN/ATAC.bed"), sep='\t', header=False, index=False)

ATAC = pybedtools.BedTool(os.path.join(DATA_DIR, "GRN/ATAC.bed"))
tss = pybedtools.BedTool(os.path.join(DATA_DIR, "GRN/marker_genes_TSS.bed"))  

overlap = tss.intersect(ATAC, wo=True)
overlap.moveto(os.path.join(DATA_DIR, "GRN/overlap.bed"))

***** WARNING: File ./SpaDC/GRN/ATAC.bed has inconsistent naming convention for record:
GL456233.1	38751	39661

***** WARNING: File ./SpaDC/GRN/ATAC.bed has inconsistent naming convention for record:
GL456233.1	38751	39661



<BedTool(./SpaDC/GRN/overlap.bed)>

### Identifying functional CREs 

In [4]:
overlap = pd.read_csv(os.path.join(DATA_DIR, "GRN/overlap.bed"), sep='\t', header=None)
print(overlap)

group = overlap.groupby([3])

gene_list = []
peak_list = []

for x, y in group:
    gene_list.append(x[0])
    peak = []
    for i in range(len(y)):
        peak.append('-'.join('%s'%a for a in y.iloc[i, 4:7]))
    peak_list.append(peak)
    
rna = sc.read_h5ad(os.path.join(DATA_DIR, "P22/ATAC/adata_RNA.h5ad"))
rna.X = rna.X.todense()

# get atac_denoise 
atac = sc.read_h5ad(os.path.join(DATA_DIR, "P22/ATAC/adata_peaks_normalized.h5ad"))
seq = pd.read_csv(os.path.join(DATA_DIR, "P22/ATAC/seqs.txt"), sep='\t')
# model.pt generated from Tutorial_P22.ipynb 
model_state_dict = './model.pt'
atac_denoise = SpaDC.get_denoise_adata(atac, seq, model_state_dict)
atac.X = atac_denoise.X

# get result cluster
result = sc.read_h5ad(os.path.join(DATA_DIR, "P22/ATAC/p22_ATAC_result.h5ad"))
cluster_annotation = {
    '1-cp':'cp', 
    '2-ctx':'ctx', 
    '3-ctx':'ctx', 
    '4-ctx':'ctx', 
    '5-ctx':'ctx', 
    '6-acb':'acb', 
    '7-ccg':'ccg', 
    '8-acb':'acb', 
    '9-ctx':'ctx', 
    '10-ls':'ls', 
    '11-aca':'aca', 
    '12-ctx':'ctx', 
    '13-cp':'cp', 
    '14-ndb':'ndb', 
    '15-aca':'aca', 
    '16-vl':'vl', 
    '17-aco':'aco', 
    '18-lpo':'lpo'
}
result.obs['SpaDC'] = result.obs['SpaDC'].map(cluster_annotation).astype('category')
atac.obs['SpaDC'] = result.obs['SpaDC'].values

gene_index =  list(rna.var_names.get_indexer(gene_list))
peak_index = []
for peak in peak_list:
    index = list(atac.var_names.get_indexer(peak))
    peak_index.append(index)

import xgboost as xgb
import shap
import os
        
CREs_list = []
all_df = pd.DataFrame()
for i in range(len(gene_index)):
    X = atac[:, peak_index[i]].X
    y = rna[:, gene_index[i]].X

    model = xgb.XGBRegressor()
    model.fit(X,y)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    shap_result = pd.DataFrame(shap_values, index=atac.obs_names, columns=peak_list[i])  
    shap_result['cluster'] = atac.obs['SpaDC'].values
    # print(shap_result)

    shap_long = shap_result.reset_index().melt(id_vars=['index', 'cluster'], var_name='peak', value_name='shapley_value')
    shap_long = shap_long.rename(columns={'index': 'cell'})
    # print(shap_long)

    df = shap_long.groupby(['cluster', 'peak'], as_index=False)['shapley_value'].mean()

    df['norm_shapley_value'] = df['shapley_value'].transform(
        lambda x: (x - x.min()) / (x.max() - x.min())
    )

    df = df[df['norm_shapley_value'] > 0.5]
    df['gene'] = gene_list[i]
    df = df[['gene', 'peak', 'cluster', 'shapley_value', 'norm_shapley_value']]

    CREs_list.extend(df['peak'].tolist())   
    all_df = pd.concat([all_df, df], ignore_index=True)

all_df.to_csv(os.path.join(DATA_DIR, "GRN/total_gene_peak.csv"), index=False)

# get CREs
unique_CREs = sorted(set(CREs_list)) 
CREs_df = pd.DataFrame(x.split('-') for x in unique_CREs)
print(CREs_df)
CREs_df.to_csv(os.path.join(DATA_DIR, "GRN/CREs.bed"), sep='\t', index=False, header=None)    

         0          1          2       3     4          5          6    7
0     chr1   39650913   40150913  Map4k4  chr1   39650601   39651430  517
1     chr1   39650913   40150913  Map4k4  chr1   39656722   39657593  871
2     chr1   39650913   40150913  Map4k4  chr1   39662969   39663811  842
3     chr1   39650913   40150913  Map4k4  chr1   39676102   39676975  873
4     chr1   39650913   40150913  Map4k4  chr1   39685563   39686412  849
...    ...        ...        ...     ...   ...        ...        ...  ...
3223  chrX  136572671  137072671    Plp1  chrX  137015061  137015959  898
3224  chrX  136572671  137072671    Plp1  chrX  137037765  137038673  908
3225  chrX  136572671  137072671    Plp1  chrX  137049100  137049992  892
3226  chrX  136572671  137072671    Plp1  chrX  137019085  137019981  896
3227  chrX  136572671  137072671    Plp1  chrX  136707734  136708386  652

[3228 rows x 8 columns]
         0          1          2
0     chr1  132431461  132432386
1     chr1  132684226

# Module 2: detecting TFs for functional CREs

#### GRN/Mus_musculus_motif_fasta and GRN/peak_motif_perturbation can be generated using the R-based methods provided in: [GRN_R](https://github.com/mcllllllll/SpaDC/blob/master/GRN)

### Generating motif-specific background distribution

In [5]:
import scanpy as sc
from model import SpaDC
from utils import *
from pathlib import Path
from tqdm import tqdm

atac = sc.read_h5ad(os.path.join(DATA_DIR, "P22/ATAC/adata_peaks_normalized.h5ad"))
print(atac)

model = SpaDC(atac.X.shape[0])
model.load_state_dict(torch.load(model_state_dict))

# get d_background
folder = Path(os.path.join(DATA_DIR, "GRN/Mus_musculus_motif_fasta/shuffled_peaks_motifs"))
fasta_files = list(folder.glob("*.fasta"))

fasta_bg = os.path.join(DATA_DIR, "GRN/Mus_musculus_motif_fasta/shuffled_peaks.fasta")

d_background_dict = {}

pred_bg= pred_on_fasta(fasta_bg, model)
for fasta_motif in tqdm(fasta_files, desc="Calculating background distribution"):
    pred_motif = pred_on_fasta(fasta_motif, model)
    d_background = pred_motif - pred_bg

    d_background_dict[fasta_motif.stem] = d_background.tolist()

AnnData object with n_obs × n_vars = 9215 × 121068
    obs: 'nCount_Spatial', 'nFeature_Spatial', 'nCount_SCT', 'nFeature_SCT', 'nCount_ATAC', 'nFeature_ATAC', 'nCount_peaks', 'nFeature_peaks', 'RNA_clusters', 'ATAC_clusters'
    var: 'count', 'percentile'
    uns: 'ATAC', 'ATAC_clusters_colors', 'umap'
    obsm: 'X_lsi', 'X_pca', 'X_umap', 'spatial'
    obsp: 'ATAC_connectivities', 'ATAC_distances'


Calculating background distribution: 100%|██████████| 153/153 [1:32:20<00:00, 36.21s/it]


### Computing TF activity via perturbation

In [6]:
result = sc.read_h5ad(os.path.join(DATA_DIR, "P22/ATAC/p22_ATAC_result.h5ad"))
cluster_annotation = {
    '1-cp':'cp', 
    '2-ctx':'ctx', 
    '3-ctx':'ctx', 
    '4-ctx':'ctx', 
    '5-ctx':'ctx', 
    '6-acb':'acb', 
    '7-ccg':'ccg', 
    '8-acb':'acb', 
    '9-ctx':'ctx', 
    '10-ls':'ls', 
    '11-aca':'aca', 
    '12-ctx':'ctx', 
    '13-cp':'cp', 
    '14-ndb':'ndb', 
    '15-aca':'aca', 
    '16-vl':'vl', 
    '17-aco':'aco', 
    '18-lpo':'lpo'
}
result.obs['SpaDC'] = result.obs['SpaDC'].map(cluster_annotation).astype('category')
atac.obs['SpaDC'] = result.obs['SpaDC'].values

# This contains only a subset of CREs
folder = Path(os.path.join(DATA_DIR, "GRN/peak_motif_perturbation"))

all_df = pd.DataFrame()
subfolders = [x for x in folder.iterdir() if x.is_dir()]
for subfolder in tqdm(subfolders, desc="Processing CREs perturbation"): 
    pred_peak = pred_on_fasta("%s/original.fasta" % subfolder, model)
    for fasta_file in subfolder.glob("*.fasta"):
        if fasta_file.name != "original.fasta":                    
            pred_motif_Perturbation_mean = pred_on_fasta(fasta_file, model).mean(axis=0)
            d_pi = pred_peak - pred_motif_Perturbation_mean
            
            tf_name = str(fasta_file.stem).replace("_Perturbation", "")
            d_pi = np.array(d_pi)
            d_background = np.array(d_background_dict[tf_name])

            p_values = np.mean(d_background >= d_pi, axis=0)
            eps = 1e-10  
            activity = -np.log(np.clip(p_values, eps, 1.0))

            activity_result = pd.DataFrame(activity.T, index=atac.obs_names, columns=[subfolder.name])  
            activity_result['cluster'] = atac.obs['SpaDC'].values

            activity_result_long = activity_result.reset_index().melt(id_vars=['index', 'cluster'], var_name='peak', value_name='activity')
            activity_result_long = activity_result_long.rename(columns={'index': 'cell'})

            df = activity_result_long.groupby(['cluster', 'peak'], as_index=False)['activity'].mean()

            df['tf'] = tf_name
            df = df[['peak', 'tf', 'cluster', 'activity']]

            all_df = pd.concat([all_df, df], ignore_index=True)

all_df.to_csv(os.path.join(DATA_DIR, "GRN/peak_tf_activity.csv"), index=False)   

Processing CREs perturbation: 100%|██████████| 189/189 [6:24:31<00:00, 122.07s/it]  


# Module 3: linking TF to target genes

### The output is the dataframe with three columns: TF, TG, and regulatory_strength

In [7]:
import pandas as pd

gene_top_peaks = pd.read_csv(os.path.join(DATA_DIR, "GRN/total_gene_peak.csv"))  
peak_tf_activity = pd.read_csv(os.path.join(DATA_DIR, "GRN/peak_tf_activity.csv"))  

merged = pd.merge(gene_top_peaks, peak_tf_activity, on=["peak", "cluster"])
print(merged)

# threshold = 0.8
# merged = merged[merged["activity"] > threshold]
# print(merged)

# activity × norm_shap
merged["weighted_activity"] = merged["activity"] * merged["norm_shapley_value"]
result = merged.groupby(["tf", "gene"])["weighted_activity"].mean().reset_index()

result.rename(columns={"weighted_activity": "regulatory_strength"}, inplace=True)
print(result)

result.to_csv(os.path.join(DATA_DIR, "GRN/tf_gene_regulatory_strength.csv"), index=False)

           gene                      peak cluster  shapley_value  \
0        Atp2b4  chr1-133655119-133655787     aca       0.039381   
1        Atp2b4  chr1-133655119-133655787     aca       0.039381   
2        Atp2b4  chr1-133655119-133655787     aca       0.039381   
3        Atp2b4  chr1-133655119-133655787     aca       0.039381   
4        Atp2b4  chr1-133655119-133655787     aca       0.039381   
...         ...                       ...     ...            ...   
23808  Syndig1l   chr12-84827324-84828204      vl       0.110050   
23809  Syndig1l   chr12-84827324-84828204      vl       0.110050   
23810  Syndig1l   chr12-84827324-84828204      vl       0.110050   
23811  Syndig1l   chr12-84827324-84828204      vl       0.110050   
23812  Syndig1l   chr12-84827324-84828204      vl       0.110050   

       norm_shapley_value                tf  activity  
0                0.505955    MA0029.2_Mecom  0.730566  
1                0.505955     MA0050.4_Irf1  0.604974  
2              